# Pakages import

In [3]:
import os
import json
import pandas as pd
import yaml
import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# apollo Scraper

data we want: lessons, teachers, type, room, date


In [5]:
with open("config.yaml","r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
print(config)
username = config['credentials']['user']
password = config['credentials']['pass']
Credentials = HTTPBasicAuth(username, password)

{'credentials': {'user': '241956', 'pass': 'Chleb@2005'}}


In [ ]:
group_id = input('input group id: ')

group_id = '252681' # --- IGNORE ---

Url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"

response = requests.get(Url, auth=Credentials)
response.encoding="utf-8"
print(response.status_code)

page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

200
<class 'bs4.BeautifulSoup'>


In [39]:
group = page_dom.select_one('div.grupa').get_text(strip=True)
print(group)

ZICSS1-1211


In [97]:
classes_tag = page_dom.select_one('table')
with open('temp.html', 'w', encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
#print(classes_tag)
classes = pd.read_html('temp.html', encoding="UTF-8")[0]
os.remove('temp.html')
#print(classes)

### filter out un-neccisery types and only keep lektorat, czwiczenia and exam

In [99]:
classes = classes.loc[classes['Typ'].isin(['lektorat', 'ćwiczenia', 'egzamin'])]
classes = classes[classes['Sala'] != 'Wybierz swoją grupę językową']

In [85]:
classes[['Day','start time', 'hyphen', 'end time', 'duration']] = classes['Dzień, godzina'].str.split(' ', expand=True)

In [86]:
classes['duration'] = classes['duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [87]:
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [88]:
classes["Sala"] = classes["Sala"].str.replace(r'Win.*', '', regex=True)

In [31]:
if not os.path.exists("schedules"):
    os.makedirs("schedules", exist_ok=True)

In [100]:
classes.to_csv(f"schedules/{group}.csv")